# LeetCode 1662 — Check If Two String Arrays are Equivalent bằng Python

Link đề: https://leetcode.com/problems/check-if-two-string-arrays-are-equivalent/

Độ khó: **Easy** · Chủ đề: Array, String

Skeleton LeetCode:

```python
class Solution:
    def arrayStringsAreEqual(self, word1: list[str], word2: list[str]) -> bool:

```

Mục tiêu học:

1. Hiểu "mảng **biểu diễn** một chuỗi" = nối các phần tử theo thứ tự.
2. Nắm `"".join(...)`, so sánh chuỗi bằng `==`, và vì sao nối bằng `+=` trong vòng lặp có thể chậm.
3. Viết lời giải ngắn nhất (nối rồi so) và hiểu chi phí bộ nhớ của nó.
4. Nâng lên **bốn con trỏ** (hai cặp `(chỉ số từ, chỉ số ký tự)`) để so từng ký tự với bộ nhớ `O(1)`.
5. Làm quen **generator** (`yield`) và `itertools.chain`, `zip_longest` để so hai "dòng ký tự" lười (lazy).
6. Tự viết lại và giải các biến thể so sánh chuỗi bị chia mảnh.

## 1. Bài toán yêu cầu gì?

Cho hai mảng chuỗi `word1` và `word2`. Một mảng **biểu diễn** chuỗi tạo bởi việc nối các phần tử theo thứ tự.

Trả về `True` nếu hai mảng biểu diễn **cùng một chuỗi**, ngược lại `False`.

| word1 | word2 | Chuỗi 1 | Chuỗi 2 | Kết quả |
|---|---|---|---|---|
| `["ab","c"]` | `["a","bc"]` | `"abc"` | `"abc"` | `True` |
| `["a","cb"]` | `["ab","c"]` | `"acb"` | `"abc"` | `False` |
| `["abc","d","defg"]` | `["abcddefg"]` | `"abcddefg"` | `"abcddefg"` | `True` |

Constraints:

- `1 <= word1.length, word2.length <= 10^3`
- `1 <= word1[i].length, word2[i].length <= 10^3`
- `1 <= sum(word1[i].length), sum(word2[i].length) <= 10^3`
- Chỉ chữ cái thường.

Điểm quan trọng: **cách cắt mảnh không quan trọng**, chỉ quan trọng chuỗi sau khi nối.

In [1]:
word1 = ["abc", "d", "defg"]
word2 = ["abcddefg"]
s1 = ""
for part in word1:
    s1 = s1 + part
    print("nối thêm", repr(part), "->", repr(s1))
print("word2 nối lại:", repr("".join(word2)))
print("bằng nhau?", s1 == "".join(word2))

nối thêm 'abc' -> 'abc'
nối thêm 'd' -> 'abcd'
nối thêm 'defg' -> 'abcddefg'
word2 nối lại: 'abcddefg'
bằng nhau? True


## 2. Nền tảng Python cần biết

### 2.1. `"".join(list_chuỗi)`

`sep.join(xs)` nối các chuỗi trong `xs`, chèn `sep` vào giữa. Với `sep = ""` thì nối liền.

In [2]:
print("".join(["ab", "c"]))
print("-".join(["ab", "c"]))
print(repr("".join([])), "(list rỗng -> chuỗi rỗng)")

abc
ab-c
'' (list rỗng -> chuỗi rỗng)


### 2.2. So sánh chuỗi bằng `==`

Hai chuỗi bằng nhau khi cùng độ dài và từng ký tự cùng vị trí giống nhau. Python so từ trái sang phải và **dừng sớm** ở ký tự khác đầu tiên.

In [3]:
print("abc" == "abc")
print("acb" == "abc")
print("abc" == "abcd")

True
False
False


### 2.3. Vì sao `s += part` trong vòng lặp có thể chậm?

Chuỗi Python **bất biến** (immutable): `s + part` tạo **chuỗi mới** và sao chép toàn bộ `s`. Lặp nhiều lần có thể tốn `O(L²)` (L là tổng độ dài). `"".join` tính tổng độ dài trước rồi sao chép một lần: `O(L)`.

(CPython đôi khi tối ưu `+=` tại chỗ, nhưng không nên dựa vào điều đó.)

In [4]:
s = "ab"
t = s
s += "c"
print("s =", s, "| t vẫn là", t, "-> s là chuỗi MỚI")

s = abc | t vẫn là ab -> s là chuỗi MỚI


### 2.4. Truy cập chuỗi con trong list lồng: `word[i][j]`

`word1[i]` là chuỗi thứ `i`; `word1[i][j]` là ký tự thứ `j` của chuỗi đó. Đây là nền tảng cho cách con trỏ.

In [5]:
word1 = ["ab", "c"]
for i in range(len(word1)):
    for j in range(len(word1[i])):
        print(f"word1[{i}][{j}] = {word1[i][j]!r}")

word1[0][0] = 'a'
word1[0][1] = 'b'
word1[1][0] = 'c'


### 2.5. Generator và `yield`

Hàm có `yield` là **generator**: mỗi lần được hỏi, nó "nhả" ra một giá trị rồi tạm dừng, lần sau chạy tiếp. Không tạo cả list trong bộ nhớ.

In [6]:
def chars_of(words):
    for w in words:
        for ch in w:
            yield ch

g = chars_of(["ab", "c"])
print(next(g), next(g), next(g))
print(list(chars_of(["abc", "d"])))

a b c
['a', 'b', 'c', 'd']


`next(g)` lấy giá trị kế tiếp. Khi hết, `next` báo `StopIteration`; `for` và `list()` tự xử lý điều này.

## 3. Cách trực giác — nối hết rồi so sánh

Đúng như hint của đề:

```python
return "".join(word1) == "".join(word2)
```

In [7]:
def equal_join(word1: list[str], word2: list[str]) -> bool:
    return "".join(word1) == "".join(word2)

print(equal_join(["ab", "c"], ["a", "bc"]))
print(equal_join(["a", "cb"], ["ab", "c"]))
print(equal_join(["abc", "d", "defg"], ["abcddefg"]))

True
False
True


### Có "nút thắt" không?

Với constraints (tổng độ dài `<= 1000`), cách này **hoàn toàn đủ** và là lời giải nên nộp. Tuy vậy, phỏng vấn viên hay hỏi tiếp:

> "Nếu mỗi mảng biểu diễn một chuỗi **rất dài** (ví dụ hàng GB, đọc từ file), em có tạo được hai chuỗi khổng lồ không?"

Hai chuỗi nối tốn thêm `O(L1 + L2)` bộ nhớ. Hơn nữa, nếu ký tự **đầu tiên** đã khác, ta vẫn phải nối **toàn bộ** trước khi so. Đây là chỗ để cải tiến: **so trực tiếp từng ký tự, dừng sớm, không tạo chuỗi mới**.

In [8]:
import sys
w = ["x" * 1000] * 1000            # tổng độ dài 10^6
joined = "".join(w)
print("Kích thước chuỗi nối ~", sys.getsizeof(joined) // 1024, "KB")
print("List gốc chỉ giữ tham chiếu:", sys.getsizeof(w) // 1024, "KB (các phần tử dùng chung một chuỗi)")

Kích thước chuỗi nối ~ 976 KB
List gốc chỉ giữ tham chiếu: 7 KB (các phần tử dùng chung một chuỗi)


## 4. Insight — bốn con trỏ duyệt "chuỗi ảo"

Ta không cần chuỗi nối, chỉ cần đọc **ký tự kế tiếp** của mỗi bên. Vị trí trong "chuỗi ảo" của `word1` được mô tả bằng hai số:

- `i1`: đang ở từ thứ mấy.
- `j1`: đang ở ký tự thứ mấy trong `word1[i1]`.

Tương tự `i2`, `j2` cho `word2`. Mỗi bước:

1. So `word1[i1][j1]` với `word2[i2][j2]`; khác → `False`.
2. Tăng `j1`, `j2`. Nếu `j1` chạm cuối từ → `i1 += 1`, `j1 = 0` (sang từ kế). Tương tự bên 2.
3. Dừng khi một bên hết. Bằng nhau ⇔ **cả hai bên hết cùng lúc**.

In [9]:
word = ["ab", "c"]
i, j = 0, 0
while i < len(word):
    print(f"(i={i}, j={j}) -> {word[i][j]!r}")
    j += 1
    if j == len(word[i]):
        i, j = i + 1, 0
print("hết: i =", i)

(i=0, j=0) -> 'a'
(i=0, j=1) -> 'b'
(i=1, j=0) -> 'c'
hết: i = 2


## 5. Lời giải chính (tối ưu bộ nhớ) — bốn con trỏ

Giảng từng đoạn:

**Đoạn 1 — khởi tạo:** `i1 = j1 = i2 = j2 = 0`.

**Đoạn 2 — vòng lặp khi cả hai còn ký tự:** `while i1 < len(word1) and i2 < len(word2)`.

**Đoạn 3 — so ký tự:** khác thì `return False`.

**Đoạn 4 — tiến con trỏ** mỗi bên, nhảy sang từ kế khi hết từ.

**Đoạn 5 — sau vòng:** cả hai phải cùng hết: `return i1 == len(word1) and i2 == len(word2)`.

In [10]:
def equal_pointers(word1: list[str], word2: list[str]) -> bool:
    i1 = j1 = i2 = j2 = 0
    while i1 < len(word1) and i2 < len(word2):
        if word1[i1][j1] != word2[i2][j2]:
            return False
        j1 += 1
        if j1 == len(word1[i1]):
            i1, j1 = i1 + 1, 0
        j2 += 1
        if j2 == len(word2[i2]):
            i2, j2 = i2 + 1, 0
    return i1 == len(word1) and i2 == len(word2)

print(equal_pointers(["ab", "c"], ["a", "bc"]))
print(equal_pointers(["a", "cb"], ["ab", "c"]))
print(equal_pointers(["abc", "d", "defg"], ["abcddefg"]))

True
False
True


`i1 = j1 = i2 = j2 = 0` là **gán nhiều biến cùng giá trị** (an toàn với số vì số bất biến).

Vì sao điều kiện cuối cần **cả hai** hết? Nếu `word1 = ["ab"]`, `word2 = ["abc"]`, vòng lặp dừng khi `word1` hết nhưng `word2` còn `"c"` → phải trả `False`.

## 6. TRACE từng bước

Trace `word1 = ["ab", "c"]`, `word2 = ["a", "bc"]`:

| bước | (i1,j1) | ký tự 1 | (i2,j2) | ký tự 2 | bằng? | sau khi tiến |
|---:|---|:---:|---|:---:|:---:|---|
| 1 | (0,0) | a | (0,0) | a | ✔ | (0,1), (1,0) — bên 2 sang từ mới |
| 2 | (0,1) | b | (1,0) | b | ✔ | (1,0), (1,1) — bên 1 sang từ mới |
| 3 | (1,0) | c | (1,1) | c | ✔ | (2,0), (2,0) |

Cả hai `i = 2 = len` → `True`.

In [11]:
def trace_pointers(word1: list[str], word2: list[str]) -> bool:
    i1 = j1 = i2 = j2 = 0
    step = 1
    while i1 < len(word1) and i2 < len(word2):
        c1, c2 = word1[i1][j1], word2[i2][j2]
        same = c1 == c2
        print(f"bước {step}: ({i1},{j1})={c1!r} vs ({i2},{j2})={c2!r} -> {'bằng' if same else 'KHÁC'}")
        if not same:
            return False
        j1 += 1
        if j1 == len(word1[i1]):
            i1, j1 = i1 + 1, 0
        j2 += 1
        if j2 == len(word2[i2]):
            i2, j2 = i2 + 1, 0
        step += 1
    done = i1 == len(word1) and i2 == len(word2)
    print(f"kết thúc: i1={i1}/{len(word1)}, i2={i2}/{len(word2)} -> {done}")
    return done

trace_pointers(["ab", "c"], ["a", "bc"])

bước 1: (0,0)='a' vs (0,0)='a' -> bằng
bước 2: (0,1)='b' vs (1,0)='b' -> bằng
bước 3: (1,0)='c' vs (1,1)='c' -> bằng
kết thúc: i1=2/2, i2=2/2 -> True


True

In [12]:
trace_pointers(["a", "cb"], ["ab", "c"])
print()
trace_pointers(["ab"], ["abc"])

bước 1: (0,0)='a' vs (0,0)='a' -> bằng
bước 2: (1,0)='c' vs (0,1)='b' -> KHÁC

bước 1: (0,0)='a' vs (0,0)='a' -> bằng
bước 2: (0,1)='b' vs (0,1)='b' -> bằng
kết thúc: i1=1/1, i2=0/1 -> False


False

## 7. Code nộp LeetCode

Bản ngắn gọn (nên dùng với constraints của đề):

In [13]:
class Solution:
    def arrayStringsAreEqual(self, word1: list[str], word2: list[str]) -> bool:
        return "".join(word1) == "".join(word2)

sol = Solution()
assert sol.arrayStringsAreEqual(["ab", "c"], ["a", "bc"]) is True
assert sol.arrayStringsAreEqual(["a", "cb"], ["ab", "c"]) is False
assert sol.arrayStringsAreEqual(["abc", "d", "defg"], ["abcddefg"]) is True
print("Solution (join) PASS")

Solution (join) PASS


Bản bốn con trỏ, bộ nhớ `O(1)` (dùng khi muốn thể hiện trong phỏng vấn):

In [14]:
class SolutionPointers:
    def arrayStringsAreEqual(self, word1: list[str], word2: list[str]) -> bool:
        i1 = j1 = i2 = j2 = 0
        while i1 < len(word1) and i2 < len(word2):
            if word1[i1][j1] != word2[i2][j2]:
                return False
            j1 += 1
            if j1 == len(word1[i1]):
                i1, j1 = i1 + 1, 0
            j2 += 1
            if j2 == len(word2[i2]):
                i2, j2 = i2 + 1, 0
        return i1 == len(word1) and i2 == len(word2)

sp = SolutionPointers()
assert sp.arrayStringsAreEqual(["ab", "c"], ["a", "bc"]) is True
assert sp.arrayStringsAreEqual(["a", "cb"], ["ab", "c"]) is False
assert sp.arrayStringsAreEqual(["abc", "d", "defg"], ["abcddefg"]) is True
print("SolutionPointers PASS")

SolutionPointers PASS


## 8. Các lời giải khác

### 8.1. Generator + `zip_longest`

Tạo hai generator ký tự, rồi ghép cặp bằng `itertools.zip_longest`. Khác `zip` (dừng ở dãy ngắn hơn), `zip_longest` điền `fillvalue` (mặc định `None`) cho dãy ngắn — nhờ đó phát hiện được độ dài khác nhau.

In [15]:
from itertools import zip_longest

print(list(zip("ab", "abc")))
print(list(zip_longest("ab", "abc")))

def equal_generators(word1: list[str], word2: list[str]) -> bool:
    def chars(words):
        for w in words:
            yield from w          # nhả lần lượt từng ký tự của w
    return all(a == b for a, b in zip_longest(chars(word1), chars(word2)))

print(equal_generators(["ab", "c"], ["a", "bc"]))
print(equal_generators(["ab"], ["abc"]))

[('a', 'a'), ('b', 'b')]
[('a', 'a'), ('b', 'b'), (None, 'c')]
True
False


`yield from w` tương đương `for ch in w: yield ch`. `all(...)` dừng ngay ở cặp khác đầu tiên, nên vẫn **dừng sớm** như bốn con trỏ.

### 8.2. `itertools.chain`

`chain(*word1)` trải các chuỗi thành một dòng ký tự liên tục — chính là generator ở trên, có sẵn trong thư viện.

In [16]:
from itertools import chain, zip_longest

print(list(chain("ab", "c")))

def equal_chain(word1: list[str], word2: list[str]) -> bool:
    return all(a == b for a, b in zip_longest(chain(*word1), chain(*word2)))

print(equal_chain(["abc", "d", "defg"], ["abcddefg"]))
print(equal_chain(["a", "cb"], ["ab", "c"]))

['a', 'b', 'c']
True
False


### 8.3. So độ dài trước để loại nhanh

Nếu tổng độ dài khác nhau thì chắc chắn `False`. `sum(map(len, word1))` tính tổng độ dài mà không nối chuỗi.

In [17]:
def equal_length_first(word1: list[str], word2: list[str]) -> bool:
    if sum(map(len, word1)) != sum(map(len, word2)):
        return False
    return equal_pointers(word1, word2)

print(sum(map(len, ["abc", "d", "defg"])))
print(equal_length_first(["ab"], ["abc"]))
print(equal_length_first(["ab", "c"], ["a", "bc"]))

8
False
True


### 8.4. Bảng so sánh các cách

Gọi `L = L1 + L2` là tổng số ký tự hai bên.

| Cách | Ý tưởng | Thời gian | Bộ nhớ phụ | Dừng sớm? | Nhận xét |
|---|---|---|---|---|---|
| `join` rồi `==` | nối hai chuỗi | `O(L)` | `O(L)` | không (phải nối hết) | **ngắn nhất, nên nộp** |
| Bốn con trỏ | duyệt chuỗi ảo | `O(L)` | `O(1)` | có | tốt cho phỏng vấn |
| Generator + zip_longest | dòng ký tự lười | `O(L)` | `O(1)` | có | Pythonic |
| `chain` | generator có sẵn | `O(L)` | `O(1)` | có | gọn nhất trong nhóm lười |
| So độ dài trước | loại nhanh | `O(L)` | `O(1)` | có | thêm bước rẻ |

## 9. Vì sao bốn con trỏ đúng?

Gọi `S1`, `S2` là hai chuỗi nối (không tạo ra thật). Đặt `p` là số ký tự đã so.

**Invariant:** ở đầu mỗi vòng lặp, `S1[0:p] == S2[0:p]`, và `(i1, j1)` trỏ đúng ký tự `S1[p]` (tương tự `(i2, j2)` trỏ `S2[p]`).

- Ban đầu `p = 0`, cả hai trỏ ký tự đầu (mọi từ đều dài `>= 1` theo constraints, nên `word[0][0]` tồn tại).
- Nếu `S1[p] != S2[p]` → hai chuỗi khác nhau → `False` đúng.
- Nếu bằng, tăng `p`; logic "hết từ thì sang từ kế, `j = 0`" giữ cho `(i, j)` trỏ đúng `S[p]` mới. Vì mọi từ dài `>= 1`, không bao giờ trỏ vào từ rỗng.

Khi vòng dừng, ít nhất một bên đã hết (`p` bằng độ dài của nó). Hai chuỗi bằng nhau ⇔ phần đã so bằng nhau **và** cả hai cùng hết ⇔ điều kiện `return`.

Chú ý: nếu đề cho phép **từ rỗng** `""`, cần vòng `while` bỏ qua từ rỗng — xem lỗi 4.

## 10. Độ phức tạp

Gọi `L1`, `L2` là tổng độ dài của `word1`, `word2`.

- **join:** Time `O(L1 + L2)` (nối) + `O(min(L1, L2))` (so). Space `O(L1 + L2)`.
- **Bốn con trỏ / generator / chain:** Time `O(min(L1, L2))` trong vòng so (thêm `O(1)` kiểm tra cuối); xấu nhất `O(L1 + L2)`. Space `O(1)`.
- **So độ dài trước:** cộng thêm `O(len(word1) + len(word2))` để cộng độ dài.

Với `L <= 2000` theo đề, mọi cách đều chạy tức thì.

## 11. Bộ test

1. 3 ví dụ của đề.
2. Một phần tử mỗi bên, giống/khác.
3. Một bên là tiền tố của bên kia (khác độ dài).
4. Cùng ký tự nhưng khác thứ tự.
5. Cách cắt cực đoan: mỗi ký tự một phần tử vs. cả chuỗi một phần tử.
6. Đối chiếu tất cả các cách trên input ngẫu nhiên (cắt ngẫu nhiên cùng một chuỗi + làm hỏng ngẫu nhiên).

In [18]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"{name:<28} PASS -> {actual}")

f = equal_pointers
check("đề 1", f(["ab", "c"], ["a", "bc"]), True)
check("đề 2", f(["a", "cb"], ["ab", "c"]), False)
check("đề 3", f(["abc", "d", "defg"], ["abcddefg"]), True)
check("1 phần tử giống", f(["z"], ["z"]), True)
check("1 phần tử khác", f(["z"], ["y"]), False)
check("tiền tố (1 ngắn hơn)", f(["ab"], ["abc"]), False)
check("tiền tố (2 ngắn hơn)", f(["abc"], ["a", "b"]), False)
check("cắt từng ký tự", f(list("hello"), ["hello"]), True)
check("khác ở ký tự cuối", f(["aaaa", "b"], ["aaaa", "c"]), False)

đề 1                         PASS -> True
đề 2                         PASS -> False
đề 3                         PASS -> True
1 phần tử giống              PASS -> True
1 phần tử khác               PASS -> False
tiền tố (1 ngắn hơn)         PASS -> False
tiền tố (2 ngắn hơn)         PASS -> False
cắt từng ký tự               PASS -> True
khác ở ký tự cuối            PASS -> False


In [19]:
import random

rng = random.Random(1662)

def random_split(s: str) -> list[str]:
    cuts = sorted(rng.sample(range(1, len(s)), rng.randint(0, len(s) - 1))) if len(s) > 1 else []
    bounds = [0] + cuts + [len(s)]
    return [s[bounds[k]:bounds[k + 1]] for k in range(len(bounds) - 1)]

methods = [equal_join, equal_pointers, equal_generators, equal_chain, equal_length_first,
           Solution().arrayStringsAreEqual, SolutionPointers().arrayStringsAreEqual]
for _ in range(600):
    s = "".join(rng.choice("ab") for _ in range(rng.randint(1, 12)))
    t = s
    r = rng.random()
    if r < 0.3:
        t = s + rng.choice("ab")                   # dài hơn
    elif r < 0.6:
        k = rng.randrange(len(s))
        t = s[:k] + ("a" if s[k] == "b" else "b") + s[k + 1:]   # đổi 1 ký tự
    w1, w2 = random_split(s), random_split(t)
    expected = s == t
    for m in methods:
        assert m(w1, w2) == expected, (getattr(m, "__name__", m), w1, w2)
print("7 cách khớp nhau trên 600 input ngẫu nhiên")

7 cách khớp nhau trên 600 input ngẫu nhiên


## 12. Lỗi người mới hay gặp

### Lỗi 1: So từng phần tử của mảng thay vì chuỗi nối

Sai: `word1 == word2` — so **list**, nên `["ab","c"]` khác `["a","bc"]` dù cùng biểu diễn `"abc"`.

In [20]:
print("sai :", ["ab", "c"] == ["a", "bc"])
print("đúng:", equal_join(["ab", "c"], ["a", "bc"]))

sai : False
đúng: True


### Lỗi 2: Dùng `zip` thay vì `zip_longest`

`zip` dừng ở dãy ngắn hơn, nên `["ab"]` vs `["abc"]` bị coi là bằng nhau.

In [21]:
from itertools import chain

def wrong_zip(word1, word2):
    return all(a == b for a, b in zip(chain(*word1), chain(*word2)))

print("sai :", wrong_zip(["ab"], ["abc"]))
print("đúng:", equal_chain(["ab"], ["abc"]))

sai : True
đúng: False


### Lỗi 3: Bốn con trỏ nhưng chỉ kiểm tra một bên hết

Sai: `return i1 == len(word1)` — bỏ qua trường hợp `word2` còn dư.

In [22]:
def wrong_one_side(word1, word2):
    i1 = j1 = i2 = j2 = 0
    while i1 < len(word1) and i2 < len(word2):
        if word1[i1][j1] != word2[i2][j2]:
            return False
        j1 += 1
        if j1 == len(word1[i1]):
            i1, j1 = i1 + 1, 0
        j2 += 1
        if j2 == len(word2[i2]):
            i2, j2 = i2 + 1, 0
    return i1 == len(word1)

print("sai :", wrong_one_side(["ab"], ["abc"]))
print("đúng:", equal_pointers(["ab"], ["abc"]))

sai : True
đúng: False


### Lỗi 4: Tưởng code con trỏ chịu được từ rỗng

Đề đảm bảo mọi từ dài `>= 1`. Nếu tự mở rộng cho phép `""`, code con trỏ sẽ đọc `word[i][0]` của từ rỗng → `IndexError`. Cách `join` thì không bị.

In [23]:
try:
    equal_pointers(["", "ab"], ["ab"])
except IndexError as e:
    print("IndexError:", e)
print("join vẫn đúng:", equal_join(["", "ab"], ["ab"]))

IndexError: string index out of range
join vẫn đúng: True


Sửa (nếu cần): trước khi đọc ký tự, dùng `while i1 < len(word1) and j1 == len(word1[i1]): i1, j1 = i1 + 1, 0` để bỏ qua từ rỗng. Bài tập 6 sẽ yêu cầu viết bản này.

### Lỗi 5: Nối bằng `+=` trong vòng lặp với dữ liệu lớn

Đúng về logic nhưng có thể chậm (mục 2.3). Ưu tiên `"".join`.

## 13. Mẹo nhận dạng pattern

- "Mảng biểu diễn chuỗi bằng cách nối" → nghĩ tới `"".join`, hoặc **chuỗi ảo** với cặp `(chỉ số từ, chỉ số ký tự)`.
- So hai dãy có thể dài khác nhau → `zip_longest`, hoặc kiểm tra **cả hai cùng hết**.
- Cần tiết kiệm bộ nhớ / dừng sớm → **generator** / **con trỏ** thay vì tạo dữ liệu trung gian.
- Duyệt list lồng như một dãy phẳng → `itertools.chain` hoặc `yield from`.

Bài liên quan: 1640 Check Array Formation Through Concatenation, 1768 Merge Strings Alternately, 251 Flatten 2D Vector (cặp con trỏ trên list lồng), 408 Valid Word Abbreviation (con trỏ trên chuỗi nén).

## 14. Biến thể

### Biến thể 1: So sánh không phân biệt hoa/thường

In [24]:
def equal_ignore_case(word1, word2):
    return all(a == b for a, b in zip_longest(
        (c.lower() for c in chain(*word1)),
        (c.lower() for c in chain(*word2))))

print(equal_ignore_case(["Ab", "C"], ["a", "bc"]))

True


### Biến thể 2: Tìm vị trí ký tự khác đầu tiên (hoặc -1)

In [25]:
def first_mismatch(word1, word2):
    for p, (a, b) in enumerate(zip_longest(chain(*word1), chain(*word2))):
        if a != b:
            return p
    return -1

print(first_mismatch(["a", "cb"], ["ab", "c"]))
print(first_mismatch(["ab", "c"], ["a", "bc"]))
print(first_mismatch(["ab"], ["abc"]))

1
-1
2


### Biến thể 3: So sánh thứ tự từ điển (lexicographic) của hai chuỗi ảo

Trả `-1`, `0`, `1` như `strcmp`. Không nối chuỗi.

In [26]:
def compare_virtual(word1, word2):
    for a, b in zip_longest(chain(*word1), chain(*word2)):
        if a == b:
            continue
        if a is None:
            return -1          # bên 1 hết trước -> nhỏ hơn
        if b is None:
            return 1
        return -1 if a < b else 1
    return 0

print(compare_virtual(["ab"], ["abc"]), compare_virtual(["b"], ["a", "z"]), compare_virtual(["x"], ["x"]))

-1 1 0


## 15. Bài tập tự luyện

Các cell TODO chỉ định nghĩa hàm, không được gọi khi chạy notebook.

### Bài 1 — Dự đoán output

Không chạy code, dự đoán:

1. `"".join(["ab", "", "c"])`
2. `["a", "b"] == ["ab"]`
3. `list(zip("abc", "ab"))`
4. `equal_pointers(["abc"], ["ab", "c"])`

### Bài 2 — Tự viết bản join

In [27]:
def exercise_join(word1: list[str], word2: list[str]) -> bool:
    # TODO: một dòng
    pass

# assert exercise_join(["ab", "c"], ["a", "bc"]) is True

### Bài 3 — Generator ký tự

In [28]:
def exercise_chars(words: list[str]):
    # TODO: dùng yield để nhả từng ký tự
    pass

# assert list(exercise_chars(["ab", "c"])) == ["a", "b", "c"]

### Bài 4 — Tự viết bốn con trỏ

In [29]:
def exercise_pointers(word1: list[str], word2: list[str]) -> bool:
    # TODO 1: i1 = j1 = i2 = j2 = 0
    # TODO 2: so ký tự, tiến con trỏ, sang từ kế khi hết từ
    # TODO 3: kiểm tra cả hai cùng hết
    pass

# assert exercise_pointers(["ab", "c"], ["a", "bc"]) is True
# assert exercise_pointers(["ab"], ["abc"]) is False

### Bài 5 — Biến thể: đếm số vị trí ký tự khác nhau

Hai chuỗi ảo **cùng độ dài**. Trả về số vị trí `p` mà ký tự khác nhau (khoảng cách Hamming), không nối chuỗi.

In [30]:
def exercise_hamming(word1: list[str], word2: list[str]) -> int:
    # TODO: chain + zip, đếm a != b
    pass

# assert exercise_hamming(["ab", "c"], ["a", "bd"]) == 1
# assert exercise_hamming(["abc"], ["x", "y", "z"]) == 3

### Bài 6 — Mini challenge: bốn con trỏ chấp nhận từ rỗng

Mở rộng bốn con trỏ để `word1`, `word2` có thể chứa chuỗi rỗng `""` (thậm chí toàn chuỗi rỗng).

In [31]:
def exercise_pointers_empty(word1: list[str], word2: list[str]) -> bool:
    # TODO: trước mỗi lần đọc, bỏ qua các từ đã hết / rỗng ở cả hai bên
    pass

# assert exercise_pointers_empty(["", "ab", "", "c"], ["a", "", "bc"]) is True
# assert exercise_pointers_empty(["", ""], []) is True
# assert exercise_pointers_empty([""], ["a"]) is False

## 16. Gợi ý 3 mức

**Bài 1**
- Mức 1: chuỗi rỗng không thêm ký tự nào.
- Mức 2: so list là so từng phần tử.
- Mức 3: `zip` dừng ở dãy ngắn.

**Bài 2**
- Mức 1: hint của đề.
- Mức 2: `"".join(...)` hai bên.
- Mức 3: `return "".join(word1) == "".join(word2)`.

**Bài 3**
- Mức 1: hai vòng for lồng nhau.
- Mức 2: `yield ch` ở vòng trong.
- Mức 3: hoặc `yield from w`.

**Bài 4**
- Mức 1: cặp `(i, j)` cho mỗi bên.
- Mức 2: `j += 1; if j == len(word[i]): i, j = i + 1, 0`.
- Mức 3: `return i1 == len(word1) and i2 == len(word2)`.

**Bài 5**
- Mức 1: cùng độ dài nên `zip` đủ.
- Mức 2: `sum(1 for a, b in zip(...) if a != b)`.
- Mức 3: hai bên đều dùng `chain(*words)`.

**Bài 6**
- Mức 1: viết hàm phụ `skip(words, i, j)` trả `(i, j)` mới, nhảy qua các từ mà `j == len(words[i])`.
- Mức 2: gọi `skip` cho cả hai bên ở **đầu** mỗi vòng lặp.
- Mức 3: `while True`: skip; nếu một bên hết thì trả `cả hai cùng hết`; ngược lại so và tăng `j`.

## 17. Đáp án tham khảo

**Bài 1:** `"abc"`; `False`; `[('a','a'), ('b','b')]`; `True`.

In [32]:
from itertools import chain

def answer_join(word1, word2):
    return "".join(word1) == "".join(word2)

def answer_chars(words):
    for w in words:
        for ch in w:
            yield ch

def answer_pointers(word1, word2):
    i1 = j1 = i2 = j2 = 0
    while i1 < len(word1) and i2 < len(word2):
        if word1[i1][j1] != word2[i2][j2]:
            return False
        j1 += 1
        if j1 == len(word1[i1]):
            i1, j1 = i1 + 1, 0
        j2 += 1
        if j2 == len(word2[i2]):
            i2, j2 = i2 + 1, 0
    return i1 == len(word1) and i2 == len(word2)

def answer_hamming(word1, word2):
    return sum(1 for a, b in zip(chain(*word1), chain(*word2)) if a != b)

def answer_pointers_empty(word1, word2):
    def skip(words, i, j):
        while i < len(words) and j == len(words[i]):
            i, j = i + 1, 0
        return i, j
    i1 = j1 = i2 = j2 = 0
    while True:
        i1, j1 = skip(word1, i1, j1)
        i2, j2 = skip(word2, i2, j2)
        end1, end2 = i1 == len(word1), i2 == len(word2)
        if end1 or end2:
            return end1 and end2
        if word1[i1][j1] != word2[i2][j2]:
            return False
        j1 += 1
        j2 += 1

assert "".join(["ab", "", "c"]) == "abc" and (["a", "b"] == ["ab"]) is False   # Bài 1
assert list(zip("abc", "ab")) == [("a", "a"), ("b", "b")]
assert equal_pointers(["abc"], ["ab", "c"]) is True
assert answer_join(["ab", "c"], ["a", "bc"]) is True
assert list(answer_chars(["ab", "c"])) == ["a", "b", "c"]
assert answer_pointers(["ab", "c"], ["a", "bc"]) is True
assert answer_pointers(["ab"], ["abc"]) is False
assert answer_hamming(["ab", "c"], ["a", "bd"]) == 1
assert answer_hamming(["abc"], ["x", "y", "z"]) == 3
assert answer_pointers_empty(["", "ab", "", "c"], ["a", "", "bc"]) is True
assert answer_pointers_empty(["", ""], []) is True
assert answer_pointers_empty([""], ["a"]) is False
for _ in range(300):
    w1 = [rng.choice(["", "a", "b", "ab", "ba"]) for _ in range(rng.randint(0, 4))]
    w2 = [rng.choice(["", "a", "b", "ab", "ba"]) for _ in range(rng.randint(0, 4))]
    assert answer_pointers_empty(w1, w2) == answer_join(w1, w2), (w1, w2)
print("Tất cả đáp án PASS")

Tất cả đáp án PASS


## 18. Template nhớ nhanh + checklist submit

```python
class Solution:
    def arrayStringsAreEqual(self, word1: list[str], word2: list[str]) -> bool:
        return "".join(word1) == "".join(word2)
```

Khi cần `O(1)` bộ nhớ:

```python
from itertools import chain, zip_longest
return all(a == b for a, b in zip_longest(chain(*word1), chain(*word2)))
```

Câu thần chú: **"Nối rồi so — hoặc đi trên chuỗi ảo bằng (từ, ký tự) và nhớ kiểm tra cả hai cùng hết."**

Checklist trước khi submit:

- [ ] So **chuỗi nối**, không so hai list?
- [ ] Dùng `"".join` thay vì `+=` lặp?
- [ ] Nếu dùng zip: là `zip_longest`, không phải `zip`?
- [ ] Nếu dùng con trỏ: kiểm tra **cả hai** cùng hết ở cuối?
- [ ] Đã thử ca tiền tố như `["ab"]` vs `["abc"]`?